## Baseline modeling - LogisticRegression

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix, classification_report, precision_recall_curve, auc

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

In [7]:
PROJECT_ROOT = '../../'
import os

# -----------------------------
# 1. 데이터 불러오기
# -----------------------------
csv_path = os.path.join(PROJECT_ROOT, '00_data', '01_interim', 'netflix_users_train.csv')
data_df = pd.read_csv(csv_path)
data_df = pd.read_csv(csv_path, low_memory=False)

#최종 dataframe
final_df = data_df.drop('user_id', axis=1)
final_df.head(2)

,age,plan_tier,is_active,monthly_spend,age_group,subscription_tenure_days,watch_count,unique_movies,total_watch_time,avg_watch_time,watch_days,recent_watch_count,days_since_last_watch,avg_progress,completion_rate,download_ratio,avg_rating
0,55.0,1,1,11.17,5,527,20,19,859.5,42.975000,19,3.0,9,43.225000,0.1,0.3,3.714286
1,40.0,1,0,7.13,4,695,7,7,528.0,75.428571,6,0.0,126,40.971429,0.0,0.0,0.000000


In [8]:
# -----------------------------
# 2. target / feature 분리
# -----------------------------
X = final_df.drop('is_active', axis=1)
y = final_df['is_active']


# -----------------------------
# 3. Train / Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# -----------------------------
# 4. Pipeline (Scaling + Logistic)
# -----------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])


# -----------------------------
# 5. CV 설정
# -----------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# -----------------------------
# 6. 평가 지표 설정
# -----------------------------
scoring = {
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}


# -----------------------------
# 7. Cross Validation 수행
# -----------------------------
cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)


# -----------------------------
# 8. CV 결과 출력
# -----------------------------
print("CV Recall:", cv_results['test_recall'].mean())
print("CV F1-score:", cv_results['test_f1'].mean())
print("CV ROC-AUC:", cv_results['test_roc_auc'].mean())
print("CV PR-AUC:", cv_results['test_pr_auc'].mean())


# -----------------------------
# 9. 모델 학습
# -----------------------------
pipeline.fit(X_train, y_train)


# -----------------------------
# 10. 테스트 예측
# -----------------------------
pred = pipeline.predict(X_test)


# -----------------------------
# 11. 테스트 평가
# -----------------------------
print("\nClassification Report")
print(classification_report(y_test, pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, pred))

CV Recall: 0.8053537284894837
CV F1-score: 0.8315533135999008
CV ROC-AUC: 0.9275495663684934
CV PR-AUC: 0.8725777231155977

Classification Report
              precision    recall  f1-score   support

           0       0.88      0.91      0.90      1071
           1       0.85      0.80      0.83       654

    accuracy                           0.87      1725
   macro avg       0.87      0.86      0.86      1725
weighted avg       0.87      0.87      0.87      1725

Confusion Matrix
[[979  92]
 [129 525]]
